# NLP and Sequence Modeling Mini Project

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

nltk.download('punkt')
nltk.download('stopwords')


## Load Dataset

In [ ]:
df = pd.read_csv('your_dataset.csv')

df.head()


## Dataset Understanding

In [ ]:
# Number of records
print("Number of records:", len(df))

# Dataset columns
print(df.columns)


In [ ]:
text_column = 'text'
target_column = 'label'

# Unique labels
print(df[target_column].unique())

# Sample records
print(df[text_column].head())


In [ ]:
# Average text length
df['text_length'] = df[text_column].apply(len)

print("Average text length:", df['text_length'].mean())


In [ ]:
# Class distribution
print(df[target_column].value_counts())

sns.countplot(x=df[target_column])
plt.title("Class Distribution")
plt.show()


## Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    tokens = word_tokenize(text)
    
    tokens = [word for word in tokens if word not in stop_words]
    
    return " ".join(tokens)

df['clean_text'] = df[text_column].apply(clean_text)

df[['clean_text']].head()


## TF-IDF Vectorization

In [ ]:
X = df['clean_text']
y = df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


## Logistic Regression Baseline Model

In [ ]:
model = LogisticRegression()

model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)


In [ ]:
print(classification_report(y_test, y_pred))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d')

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.savefig("results/model_evaluation.png")

plt.show()


In [ ]:
predictions = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred
})

predictions.to_csv("results/sample_predictions.txt", index=False)


## Sequence Model Using LSTM

In [ ]:
tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)


In [ ]:
max_length = 100

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length)


In [ ]:
lstm_model = Sequential()

lstm_model.add(
    Embedding(
        input_dim=5000,
        output_dim=64,
        input_length=max_length
    )
)

lstm_model.add(LSTM(64))

lstm_model.add(Dense(1, activation='sigmoid'))


In [ ]:
lstm_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


In [ ]:
history = lstm_model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)


In [ ]:
loss, accuracy = lstm_model.evaluate(
    X_test_pad,
    y_test
)

print("Test Accuracy:", accuracy)


# Attention and Transformer Reflection

## Why RNNs struggle with long-term dependencies
RNNs suffer from vanishing gradients, making it difficult to remember earlier information in long sequences.

## How LSTMs help
LSTMs use memory cells and gating mechanisms to retain important information for longer periods.

## What attention solves
Attention allows the model to focus on important words in the sequence instead of compressing everything into one fixed vector.

## Why transformers are important
Transformers use self-attention and parallel processing, making them faster and more effective for modern NLP applications and Generative AI systems.
